# 01 — Scatter / Gather on Paged KV Caches

This notebook explores the low-level operations needed to read from and write
to vLLM's paged KV cache: **scatter** (writing dense K/V tensors into
non-contiguous cache pages) and **gather** (reading them back out).

These operations are the foundation for any KV cache compression scheme
inside vLLM. The idea is simple: if we can gather the full cached context
into a dense buffer, apply a compression pass (like KeyDiff's
key-similarity scoring), and scatter the survivors back — without losing
data — then we have a working compression primitive.

We start by testing that the round trip is **lossless**: data scattered
into the cache can be gathered back identically. Then we test a
**compact** operation that simulates compression by keeping a per-head
subset of tokens.

All operations target the classic paged attention KV cache layout
(5D keys, 4D values). Adapting these for the Flash Attention 2 layout
used on NVIDIA GPUs is a natural next step.

## Imports and Setup

In [ ]:
import torch
from vllm import _custom_ops as ops

torch.manual_seed(42)
torch.set_grad_enabled(False)

assert torch.cuda.is_available(), "CUDA GPU required"
print(f"GPU: {torch.cuda.get_device_name(0)}")

## Configuration

In [ ]:
DTYPE = torch.float16
NUM_KV_HEADS = 4
HEAD_SIZE = 128
BLOCK_SIZE = 16
SEQ_LEN = 137  # intentionally not a multiple of BLOCK_SIZE
DEVICE = "cuda"

## Paged KV Cache Layout

vLLM's classic paged attention stores keys and values in a block-based
format optimized for the paged attention kernel:

- **Key cache:** `[num_blocks, num_kv_heads, head_size // x, block_size, x]`
- **Value cache:** `[num_blocks, num_kv_heads, head_size, block_size]`

where `x = 16 // element_size` is a packing factor (8 for float16/bfloat16).

Keys use a 5D layout because the paged attention kernel reads them in
packed chunks of `x` elements for memory coalescing. Values use a simpler
4D layout since they are accessed differently during the attention
computation.

Each **block** holds `block_size` token slots. A **block table** maps
logical block indices (sequential) to physical block indices (which may
be scattered across GPU memory).

In [ ]:
def create_kv_caches(num_blocks, block_size, num_kv_heads, head_size, dtype,
                     device="cuda"):
    """Create KV caches in the classic paged attention layout."""
    x = 16 // torch.tensor([], dtype=dtype).element_size()
    key_cache = torch.randn(
        num_blocks, num_kv_heads, head_size // x, block_size, x,
        dtype=dtype, device=device,
    )
    value_cache = torch.randn(
        num_blocks, num_kv_heads, head_size, block_size,
        dtype=dtype, device=device,
    )
    return key_cache, value_cache


num_blocks = (SEQ_LEN + BLOCK_SIZE - 1) // BLOCK_SIZE + 4
key_cache, value_cache = create_kv_caches(
    num_blocks, BLOCK_SIZE, NUM_KV_HEADS, HEAD_SIZE, DTYPE, DEVICE,
)

print(f"Key cache shape:   {key_cache.shape}")
print(f"Value cache shape: {value_cache.shape}")
print(f"Blocks allocated:  {num_blocks}")

## Scatter / Gather Functions

Four functions that operate on the paged cache:

1. **`build_slot_mapping_for_positions`** — converts token positions to flat
   slot indices using the block table
2. **`gather_from_paged_cache`** — reads from the 5D/4D paged layout into
   dense `[num_tokens, num_kv_heads, head_size]` tensors
3. **`select_per_head`** — picks a different subset of tokens for each KV head
   (simulating per-head compression scoring)
4. **`select_per_head_randomly`** — placeholder random selection for testing

In [ ]:
def build_slot_mapping_for_positions(block_table, positions, block_size):
    """Map token positions to flat slot indices in the paged cache.

    Each slot identifies the cell dedicated to storing a token's K/V data
    across all KV heads. The same block_idx and offset address the
    corresponding entries in both the 5D key cache and the 4D value cache.
    """
    logical_block_indices = positions // block_size
    physical_block_indices = block_table[logical_block_indices]
    offsets = positions % block_size
    return physical_block_indices * block_size + offsets


def gather_from_paged_cache(key_cache, value_cache, slot_mapping,
                            num_kv_heads, head_size, block_size):
    """Read tokens from the paged cache into dense
    [num_tokens, num_kv_heads, head_size] tensors."""
    block_indices = slot_mapping // block_size
    offsets = slot_mapping % block_size

    keys_packed = key_cache[block_indices, :, :, offsets, :]
    keys = keys_packed.reshape(-1, num_kv_heads, head_size)

    values = value_cache[block_indices, :, :, offsets]

    return keys, values


def select_per_head(dense, kept_indices):
    """Select tokens per head from a dense
    [seq_len, num_kv_heads, head_size] tensor.

    kept_indices: [num_kv_heads, compacted_len] — which positions to keep
    per head. Returns: [compacted_len, num_kv_heads, head_size]
    """
    head_size = dense.shape[2]
    by_head = dense.permute(1, 0, 2)
    idx = kept_indices.unsqueeze(-1).expand(-1, -1, head_size)
    kept = by_head.gather(1, idx)
    return kept.permute(1, 0, 2).contiguous()


def select_per_head_randomly(seq_len, compacted_len, num_kv_heads, device):
    """Choose random token positions to keep per head.

    Placeholder for a real scoring algorithm (e.g. KeyDiff).
    Returns: [num_kv_heads, compacted_len] — sorted indices per head.
    """
    return torch.stack([
        torch.randperm(seq_len, device=device)[:compacted_len].sort().values
        for _ in range(num_kv_heads)
    ])


print("Functions defined")

## Experiment 1 — Scatter → Gather → Scatter Round Trip

The most basic invariant: if we write tokens into the cache and read
them back, we should get the exact same data. And if we write that
data back again, the cache should be unchanged.

This is not a trivial property — the key cache uses a packed 5D layout,
so gather must correctly unpack the `x`-strided inner dimension and
reassemble the full `head_size` vector.

In [ ]:
keys_original = torch.randn(
    SEQ_LEN, NUM_KV_HEADS, HEAD_SIZE, dtype=DTYPE, device=DEVICE,
)
values_original = torch.randn(
    SEQ_LEN, NUM_KV_HEADS, HEAD_SIZE, dtype=DTYPE, device=DEVICE,
)

num_seq_blocks = (SEQ_LEN + BLOCK_SIZE - 1) // BLOCK_SIZE
block_table = torch.arange(num_seq_blocks, dtype=torch.long, device=DEVICE)

positions = torch.arange(SEQ_LEN, dtype=torch.long, device=DEVICE)
slot_mapping = build_slot_mapping_for_positions(
    block_table, positions, BLOCK_SIZE,
)

print(f"Sequence length:   {SEQ_LEN}")
print(f"Blocks used:       {num_seq_blocks}")
print(f"Slot mapping:      [{slot_mapping[0].item()}, "
      f"{slot_mapping[1].item()}, ..., {slot_mapping[-1].item()}]")
print(f"Keys shape:        {keys_original.shape}")
print(f"Values shape:      {values_original.shape}")

In [ ]:
k_scale = torch.tensor(1.0, dtype=torch.float32, device=DEVICE)
v_scale = torch.tensor(1.0, dtype=torch.float32, device=DEVICE)

ops.reshape_and_cache(
    keys_original, values_original,
    key_cache, value_cache,
    slot_mapping, "auto", k_scale, v_scale,
)
key_cache_snapshot = key_cache.clone()
value_cache_snapshot = value_cache.clone()

print("Scattered original data into cache")

In [ ]:
keys_gathered, values_gathered = gather_from_paged_cache(
    key_cache, value_cache, slot_mapping,
    NUM_KV_HEADS, HEAD_SIZE, BLOCK_SIZE,
)

torch.testing.assert_close(keys_gathered, keys_original, atol=0, rtol=0)
torch.testing.assert_close(values_gathered, values_original, atol=0, rtol=0)

print("Gathered keys and values match original")

In [ ]:
ops.reshape_and_cache(
    keys_gathered, values_gathered,
    key_cache, value_cache,
    slot_mapping, "auto", k_scale, v_scale,
)

torch.testing.assert_close(
    key_cache, key_cache_snapshot, atol=0, rtol=0,
)
torch.testing.assert_close(
    value_cache, value_cache_snapshot, atol=0, rtol=0,
)

print("Cache unchanged after round trip — scatter/gather is lossless")

## Experiment 2 — Compact with Per-Head Token Selection

Now we test the full compression primitive. The idea:

1. **Gather** all cached tokens into a dense buffer
2. **Select** a subset of tokens — independently per KV head (simulating a
   compression algorithm like KeyDiff that scores and keeps the most
   important tokens per head)
3. **Scatter** the kept tokens back into the first `compacted_len` slots

After compaction, only the first `compacted_len` positions in the cache
are valid. The block table stays the same, but the sequence length
tracked by the scheduler would shrink.

Here we use random selection (each head keeps a random subset). In a real
compression pass, `kept_indices` would come from the scoring algorithm.

In [ ]:
COMPRESSION_RATIO = 0.5
compacted_len = int(SEQ_LEN * (1 - COMPRESSION_RATIO))

key_cache, value_cache = create_kv_caches(
    num_blocks, BLOCK_SIZE, NUM_KV_HEADS, HEAD_SIZE, DTYPE, DEVICE,
)
ops.reshape_and_cache(
    keys_original, values_original,
    key_cache, value_cache,
    slot_mapping, "auto", k_scale, v_scale,
)

kept_indices = select_per_head_randomly(SEQ_LEN, compacted_len, NUM_KV_HEADS, DEVICE)

print(f"Original sequence:  {SEQ_LEN} tokens")
print(f"Compression ratio:  {COMPRESSION_RATIO}")
print(f"Compacted length:   {compacted_len} tokens")
print(f"Kept indices shape: {kept_indices.shape}")

In [ ]:
keys_dense, values_dense = gather_from_paged_cache(
    key_cache, value_cache, slot_mapping,
    NUM_KV_HEADS, HEAD_SIZE, BLOCK_SIZE,
)
keys_compact = select_per_head(keys_dense, kept_indices)
values_compact = select_per_head(values_dense, kept_indices)

compact_positions = torch.arange(compacted_len, dtype=torch.long, device=DEVICE)
compact_slot_mapping = build_slot_mapping_for_positions(
    block_table, compact_positions, BLOCK_SIZE,
)
ops.reshape_and_cache(
    keys_compact, values_compact,
    key_cache, value_cache,
    compact_slot_mapping, "auto", k_scale, v_scale,
)

print(f"Compacted {SEQ_LEN} -> {compacted_len} tokens using random per-head selection")

## Notes and Next Steps

**Cache layout:** These experiments use the classic paged attention KV
cache layout (5D keys, 4D values) with `reshape_and_cache`. On NVIDIA
GPUs, vLLM defaults to Flash Attention 2, which uses a different layout
— both keys and values are stored as
`[num_blocks, block_size, num_kv_heads, head_size]` and use the
`reshape_and_cache_flash` kernel. Adapting gather/scatter for this
layout is a natural next step.

**What this enables:** If the round trip is lossless, inserting a
compression scoring step (like KeyDiff's key-similarity metric) between
gather and scatter should produce correct compacted caches. Experiment 2
demonstrates this pattern with random selection — replacing random
selection with actual scoring is the next piece.